# Boosting

This notebook implements a **Gradient Boosting** approach to fake news detection, following the optimization problem defined in the project specification. The objective minimises a weighted, category-balanced cross-entropy loss with regularisation:

$$Z = -\frac{1}{N}\sum_{i=1}^{N} \alpha_{c_i}\left[w_1 y_i \log(F(x_i)) + w_0(1-y_i)\log(1-F(x_i))\right] + \lambda\Omega(\Theta)$$

where $F(x) = \sum_{m=1}^{M} \beta_m f_m(x)$ is the boosting ensemble.

Project Environment: Run the code cell below to auto-detect data ingestion script. <br>
Stand-alone Environment : Skip the code cell and upload data ingestion script from github.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/project/COMP3608PROJECT')

Mounted at /content/drive


#Data Ingestion
Load the unified fake-news DataFrame using the shared `ingest_data` script. This combines three Kaggle datasets (`bhavikjikadara`, `mahdimashayekhi`, `shawkyelgendy`) and returns a DataFrame with columns: `title`, `text`, `label`, `category`, `dataset`. Basic preprocessing (null-text removal, deduplication, category normalisation) is applied inside the loader.

In [ ]:
from ingest_data import load_datasets
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...
Using Colab cache for faster access to the 'fake-news-detection' dataset.
[bhavik] Loaded 'true.csv': 21417 rows
Using Colab cache for faster access to the 'fake-news-detection' dataset.
[bhavik] Loaded 'fake.csv': 23481 rows

Loading mahdimashayekhi ...
Using Colab cache for faster access to the 'fake-news-detection-dataset' dataset.
[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...
Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'real.csv': 21869 rows
Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'fake.csv': 19999 rows

Dropped 648 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
-------------------------

#Imports

All third-party libraries that are used in this notebook are imported here for ease of reproducibility and clarity

#Text Cleaning Pipeline

Raw text from multiple sources contains noise: HTML entities, URLs, Twitter handles and other characteristics. the `clean text` essentially is normalizing each entry before vectorization.

In [ ]:
# Normalise a raw text string for TF-IDF vectorization
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)   # URLs
    s = re.sub(r'@\w+', ' ', s)                     # Twitter handles
    s = re.sub(r'<[^>]+>', ' ', s)                  # HTML tags
    s = re.sub(r'[^a-z\s]', ' ', s)                 # Non-alpha
    s = re.sub(r'\s+', ' ', s).strip()              # Whitespace
    return s

df['clean_title'] = df['title'].apply(clean_text)
df['clean_text']  = df['text'].apply(clean_text)

# Combine title + text; title is repeated to upweight headline signal
df['combined_text'] = df['clean_title'] + ' ' + df['clean_text']

print(f"Sample cleaned text:\n{df['combined_text'].iloc[0][:300]}")